In [10]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

    
import pandas as pd
from scripts.crm_leads import load_leads

In [11]:
captacao = load_leads()



Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 250 registros
Página 26: 250 registros
Página 27: 250 registros
Página 28: 250 registros
Página 29: 203 registros


In [39]:

url = 'https://docs.google.com/spreadsheets/d/13DBRdUsHT5wupKkq3Omi9VUr9Nj3R5gYVMfSSkmtXQM/export?format=xlsx'

matriculas_itabirito = pd.read_excel(url, sheet_name = "ITABIRITO")
matriculas_altamira = pd.read_excel(url, sheet_name = "ALTAMIRA")
matriculas_lorena = pd.read_excel(url, sheet_name = "LORENA")
matriculas_canaa = pd.read_excel(url, sheet_name = "Canaã")


matriculas = pd.concat([matriculas_itabirito, matriculas_altamira, matriculas_lorena, matriculas_canaa], ignore_index=True)

In [70]:
matriculas.dropna(inplace = True, subset = ['CPF (preencher apenas com números)'])

In [71]:
captacao.columns

Index(['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'Situação Funil'], dtype='str')

In [72]:
colunas_captacao = ['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'Situação Funil']
captacao = captacao[colunas_captacao]

## Mapeando leads sem proprietário e buscando na planilha

In [73]:
# leads sem proprietário

matriculas_sem_proprietario = captacao[captacao['Proprietário'].isnull() &  (captacao['Situação Funil'] == 'Matriculado') ]

In [74]:
matriculas_sem_proprietario.shape

(43, 5)

In [84]:
df = pd.merge(matriculas_sem_proprietario, matriculas, how='left', left_on='Nome Completo', right_on='NOME')

In [85]:
df[['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'NOME', 'CONSULTOR (a)', 'CPF (preencher apenas com números)']].loc[~df['CONSULTOR (a)'].isnull()].to_excel('matriculas_sem_proprietario.xlsx', index=False)

## Mapear matrículas que diverge o proprietário do consultor

In [86]:
matriculados = captacao[captacao['Situação Funil'] == 'Matriculado']
matriculados.shape

(660, 5)

In [87]:
df = matriculados.merge(matriculas,  how='left', left_on='Nome Completo', right_on='NOME')

In [88]:


proprietarios_consultores = {'Maria Eduarda Freire': 'Duda',
                             'Maria Rafaela da Rocha Tavares': 'Maria Rafaela',
                             'Adauri Marques Delfino Junior': 'Adauri',
                             'Karen Flávia': 'Flávia',
                             'Daniel Faria Santos': 'Daniel',
                             'Geovana Souza da Silva': 'Geovana',
                             'Agnata Pamela Silva Souza': 'Agnata',
                             'Giovanna Eduarda Do Carmo Braga': 'Giovanna Braga',
                             'Hailton Alves Júnior': 'Hailton',
                             'Karolaine de Oliveira Alves Souza': 'Karolaine',
                             'Thiago Oliveira da Cruz': 'Thiago',
                             'Maria Fernanda Luiz Monteiro': 'Maria Fernanda ',
                             'Ana Carolina Guimarães': 'Ana Carolina',
                             'Lorena Viana Gonçalves Pereira': 'Lorena',
                             'Thalia Gabriela de Araújo Silva': 'Thalia',
                             'Camila Guimarães Santos': 'Camila',
                             'Breno Assis Ferreira': 'Breno'}

# verificando se o proprietário é diferente do consultor
df['Proprietário'] = df['Proprietário'].map(proprietarios_consultores)

# tirando linhas onde proprietário ou consultor são nulos
df = df.dropna(subset=['Proprietário', 'CONSULTOR (a)'])
divergencias = df[df['Proprietário'] != df['CONSULTOR (a)']][['Nome Completo', 'cpf', 'ID Lead', 'Proprietário', 'NOME', 'CONSULTOR (a)', 'CPF (preencher apenas com números)']]
display(divergencias.shape)


(58, 7)

In [89]:
divergencias

,Nome Completo,cpf,ID Lead,Proprietário,NOME,CONSULTOR (a),CPF (preencher apenas com números)
28,VITORIA APARECIDA AMARAL MARCONDES,438.334.338-41,{AC3AC3B8-2A90-F111-8076-6045BD38FB26},Duda,VITORIA APARECIDA AMARAL MARCONDES,Thalia,438.334.338-41
31,Ana Paula da SIlva Coimbra,437.462.918-10,{D3C89DBD-888F-F111-8077-7CED8DB11FB7},Duda,Ana Paula da SIlva Coimbra,Maria Rafaela,437.462.918-10
82,Larissa Mirely Medeiros de Oliveira,332.783.538-11,{5E7F1C52-508C-F111-8077-6045BD39BC63},Duda,Larissa Mirely Medeiros de Oliveira,Hailton,332.783.538-11
85,Emanuella Ribeiro Freitas,052.427.802-40,{3FD972C6-2C8C-F111-8077-000D3AC0F764},Ana Carolina,Emanuella Ribeiro Freitas,Flávia,052.427.802-40
128,Alfredo Jose de Campos,339.712.318-61,{DAD599FE-4488-F111-8077-00224836BDFB},Hailton,Alfredo Jose de Campos,Adauri,339.712.318-61
140,Alan Rodrigues,094.213.286-63,{7280C10C-4E87-F111-AB0F-000D3AC0F764},Daniel,Alan Rodrigues,Giovanna Braga,094.213.286-63
149,Hildenice Gomes Barbosa da Silva,977.306.962-15,{07D5E22F-9E86-F111-AB0F-00224836B90D},Hailton,Hildenice Gomes Barbosa da Silva,Geovana,977.306.962-15
158,Karina Raissa de Amorim,488.282.728-05,{B218D78A-0386-F111-AB0F-6045BD39BE00},Maria Fernanda,Karina Raissa de Amorim,Daniel,488.282.728-05
172,Kássia Capistrano Quintas Vasconcellos,470.727.348-30,{0309EA0B-2885-F111-AB0F-6045BD39BC63},Maria Fernanda,Kássia Capistrano Quintas Vasconcellos,Duda,470.727.348-30
211,Francisca Gomes Teixeira,687.919.332-34,{8E79A3B2-3981-F111-AB0E-6045BD39BE00},Karolaine,Francisca Gomes Teixeira,Geovana,68791933234
